<a href="https://colab.research.google.com/github/juanpajedrez/learn_rag_Huggingface/blob/main/RAGAS_Video_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Library + API keys

In [63]:
!pip install --upgrade --quiet unstructured
!pip install --quiet langchain-community langchain-openai faiss-cpu
!pip install --quiet ragas

In [64]:
!pip install --upgrade ragas instructor langchain-openai

In [65]:
# urls for kubernetes documentation
urls = [
    "https://kubernetes.io/docs/concepts/overview/",
    "https://kubernetes.io/docs/concepts/architecture/",
    "https://kubernetes.io/docs/concepts/containers/",
    "https://kubernetes.io/docs/concepts/workloads/",
    "https://kubernetes.io/docs/concepts/storage/"
]

In [66]:
# download NLTK data
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [67]:
from google.colab import userdata
openai_api = userdata.get('OPENAI_API_KEY')

import os
os.environ['OPENAI_API_KEY'] = openai_api

MODEL = "gpt-5-nano"
EMBEDDING_MODEL = 'text-embedding-3-large'

# Data and RAG

In [68]:
# Import libraries, functions, classes
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

In [69]:
# Load the documents and split them
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 3000,
    chunk_overlap = 600
)

loader = UnstructuredURLLoader(urls = urls)
data = loader.load()

In [70]:
documents = text_splitter.split_documents(documents = data)

In [71]:
documents[0:1]

[Document(metadata={'source': 'https://kubernetes.io/docs/concepts/overview/'}, page_content='Overview\n\nKubernetes is a portable, extensible, open source platform for managing containerized workloads and services that facilitate both declarative configuration and automation. It has a large, rapidly growing ecosystem. Kubernetes services, support, and tools are widely available.\n\nThis page is an overview of Kubernetes.\n\nThe name Kubernetes originates from Greek, meaning helmsman or pilot. K8s as an abbreviation results from counting the eight letters between the "K" and the "s". Google open sourced the Kubernetes project in 2014. Kubernetes combines over 15 years of Google\'s experience running production workloads at scale with best-of-breed ideas and practices from the community.\n\nWhy you need Kubernetes and what it can do\n\nContainers are a good way to bundle and run your applications. In a production environment, you need to manage the containers that run the applications a

In [72]:
# Create the embeddings and store in the Database vectorstor
embeddings = OpenAIEmbeddings(model = EMBEDDING_MODEL)
vectorstore = FAISS.from_documents(documents = documents, embedding = embeddings)

In [85]:
# Perform RAG
def rag(query, k = 20):
  retrieved_docs = vectorstore.similarity_search(query, k = k)
  context = "\n".join([doc.page_content for doc in retrieved_docs])

  # Define the prompt
  prompt = f""" answer this query {query} based on the context {context}
  """

  # Call the LLM
  model = ChatOpenAI(
      model = MODEL
  )

  response = model.invoke(prompt)

  # Return the output
  return response.content, [txt.page_content for txt in retrieved_docs]

# Test the function
query = "Give me an overview of kubernetes"
answer, __ = rag(query)

In [74]:
from IPython.display import Markdown, display
display(Markdown(answer))

Here’s a concise overview of Kubernetes based on the provided context:

What Kubernetes is
- An open-source, portable, extensible platform for managing containerized workloads and services.
- It enables declarative configuration and automation, with a large ecosystem of services, tools, and support.

Why you’d use Kubernetes
- Keeps applications available with minimal downtime.
- Manages scaling and failover, supports automated deployments, and provides deployment patterns (e.g., canary).
- Handles essential platform capabilities: service discovery, load balancing, storage orchestration, and automated rollouts/rollbacks.
- Keeps resources efficiently utilized (bin packing, CPU/memory requests and limits).
- Provides self-healing (restarting/replacing unhealthy containers), secret/config management, and batch processing.
- Supports horizontal scaling, multi-architecture networking (IPv4/IPv6), and extensibility for custom needs.

What Kubernetes is not
- Not a traditional all-in-one PaaS. It provides building blocks and integrations, not a fixed, monolithic platform.
- It does not deploy source code or implement CI/CD by itself.
- It does not include built-in application-level services (e.g., databases, message buses) as core features, though these can run on Kubernetes.
- It does not mandate a single logging/monitoring solution; it offers integrations and mechanisms to collect/export metrics.
- It does not enforce a single configuration language; it exposes a declarative API that can be targeted by various declaration formats.

Key capabilities (high level)
- Service discovery and load balancing: expose containers via DNS or IP, with traffic distribution.
- Storage orchestration: auto-mounts storage from local or cloud providers.
- Automated rollouts and rollbacks: declare the desired state and Kubernetes adjusts the actual state gradually.
- Automatic bin packing: schedules containers onto nodes based on resource requirements.
- Self-healing: restarts/replaces failing containers and only exposes healthy ones.
- Secret and configuration management: manage sensitive data and config without rebuilding images.
- Batch execution: supports batch/CI workloads and task automation.
- Horizontal scaling: scale manually, via UI, or automatically based on metrics.
- Networking: supports IPv4/IPv6, dual-stack, and flexible network plugins.
- Extensibility: add features via custom resources, CRDs, and integrations with cloud providers or runtimes.

Core architecture (high level)
- Control plane: makes global decisions and manages cluster state.
  - kube-apiserver: the API front end.
  - etcd: the backing store for cluster data.
  - kube-scheduler: assigns Pods to nodes.
  - kube-controller-manager: runs control loops (e.g., node lifecycle, jobs, endpoints).
  - cloud-controller-manager: (when applicable) integrates with cloud provider APIs.
- Nodes (worker machines): run the workload
  - kubelet: agent on each node that ensures containers in Pods are running.
  - kube-proxy: handles networking for Services on the node (optional if alternative network solutions are used).
  - Container runtime: executes containers (e.g., containerd, CRI-O) via the Container Runtime Interface (CRI).
- Workloads and primitives
  - Pods: smallest deployable units; container(s) inside a Pod share the same network/IPC/storage context.
  - Workload controllers: Deployment/ReplicaSet (stateless), StatefulSet (stateful with stable identities), DaemonSet (runs on every node or selected nodes), Job/CronJob (one-off or scheduled tasks).
  - API-driven configuration: declarative objects stored in the cluster, reconciled to the desired state automatically.
- Addons and extensions
  - DNS, dashboard UI, metrics/logging collectors, network plugins, and other cluster-level features (often in kube-system namespace).

Notes on operation and customization
- Kubernetes is designed to be modular and pluggable. You can choose different components, add-ons, and runtimes, and you can implement custom scheduling or extend the API with custom resources.
- It supports multiple deployment models and scales from small to large deployments. Control plane components can be distributed for high availability.
- It integrates with various cloud providers and on-prem environments via cloud-controller-manager and other integrations.

Quick pointers to explore further
- Kubernetes components, API, and kubectl for interacting with clusters.
- Cluster architecture (control plane vs. nodes) and how workloads are scheduled.
- Workload resources (Deployments, StatefulSets, DaemonSets, Jobs, CronJobs).
- How storage, networking, and addons are implemented in typical clusters.

If you want, I can tailor this into a shorter one-page summary or focus on a specific area (e.g., architecture, workloads, or extensibility).

# Generate Synthethic Data

**6/28/2026:** Quick fix due to old `langchain_community.chat_models.vertexai` llm, link here: https://github.com/vibrantlabsai/ragas/issues/2753

In [75]:
import sys
import types

dummy_chat = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat

import langchain_community.llms
langchain_community.llms.VertexAI = type("VertexAI", (object,), {})

In [76]:
# Import classes
from ragas.llms import llm_factory
from ragas.testset import TestsetGenerator
from openai import OpenAI
from ragas.embeddings.base import embedding_factory

In [77]:
# Connect to OpenAI
client = OpenAI(api_key = openai_api)

In [81]:
# LLMS and embeddings
generator_llm = llm_factory(
    model = "gpt-4o",
    client = client
)

In [82]:
# Generator embeddings
generator_embeddings = embedding_factory(
    "openai",
    model = EMBEDDING_MODEL,
    client = client
)

In [83]:
# Remove empty chunks
chunks = [doc for doc in documents if doc.page_content and doc.page_content.strip()]

# Test Set Generation
generator = TestsetGenerator(
    llm = generator_llm,
    embedding_model = generator_embeddings
)
dataset = generator.generate_with_chunks(
    chunks = chunks,
    testset_size = 30,
    raise_exceptions=False
)

Applying SummaryExtractor:   0%|          | 0/12 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/12 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/12 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/ragas/testset/transforms/base.py:198: UserWarning: Using sync embedding model OpenAIEmbeddings in async context. This may impact performance. Consider using an async-compatible embedding model for better performance.
  property_name, property_value = await self.extract(node)


Applying ThemesExtractor:   0%|          | 0/12 [00:00<?, ?it/s]

...

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/32 [00:00<?, ?it/s]

In [84]:
# Transform the dataset
test_df = dataset.to_pandas()
test_df.head()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,Why they call Kubernetes Greek and what it do ...,"[Overview\n\nKubernetes is a portable, extensi...","The name Kubernetes originates from Greek, mea...",Cloud Infrastructure Engineer,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,What is Ceph in Kubernetis?,[Self-healing Kubernetes restarts containers t...,Kubernetes does not provide cluster storage sy...,Cloud Infrastructure Engineer,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
2,How does Kubernetes handle configuration and o...,"[Does not dictate logging, monitoring, or aler...",Kubernetes does not dictate or mandate a speci...,Cloud Infrastructure Engineer,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
3,How does RHEL work with containers and virtual...,[Virtualization allows better utilization of r...,"RHEL, as an OS distribution, supports containe...",Cloud Infrastructure Engineer,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
4,What role do Pods play in a Kubernetes cluster...,[Cluster Architecture\n\nThe architectural con...,"In a Kubernetes cluster architecture, Pods are...",Cloud Infrastructure Engineer,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer


In [86]:
from tqdm import tqdm

# Answer each query and put it in our df
answer_gen = []
context_gen = []
for query in tqdm(test_df["user_input"], desc = "Generating answers from synthethic dataset...", total = len(test_df["user_input"])):
  answer, context = rag(query)
  answer_gen.append(answer)
  context_gen.append(context)

test_df["answer_gen"] = answer_gen
test_df["context_gen"] = context_gen
test_df.head()

Generating answers from synthethic dataset...: 100%|██████████| 32/32 [09:07<00:00, 17.11s/it]


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name,answer_gen,context_gen
0,Why they call Kubernetes Greek and what it do ...,"[Overview\n\nKubernetes is a portable, extensi...","The name Kubernetes originates from Greek, mea...",Cloud Infrastructure Engineer,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer,Here's a concise answer based on the overview:...,"[Overview\n\nKubernetes is a portable, extensi..."
1,What is Ceph in Kubernetis?,[Self-healing Kubernetes restarts containers t...,Kubernetes does not provide cluster storage sy...,Cloud Infrastructure Engineer,MISSPELLED,SHORT,single_hop_specific_query_synthesizer,Ceph is a cluster storage system (external to ...,"[Overview\n\nKubernetes is a portable, extensi..."
2,How does Kubernetes handle configuration and o...,"[Does not dictate logging, monitoring, or aler...",Kubernetes does not dictate or mandate a speci...,Cloud Infrastructure Engineer,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer,Short answer\n- Kubernetes treats configuratio...,"[Does not dictate logging, monitoring, or aler..."
3,How does RHEL work with containers and virtual...,[Virtualization allows better utilization of r...,"RHEL, as an OS distribution, supports containe...",Cloud Infrastructure Engineer,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer,Short answer\nRed Hat Enterprise Linux (RHEL) ...,[Virtualization allows better utilization of r...
4,What role do Pods play in a Kubernetes cluster...,[Cluster Architecture\n\nThe architectural con...,"In a Kubernetes cluster architecture, Pods are...",Cloud Infrastructure Engineer,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer,- The Pod is the smallest deployable unit in K...,[Cluster Architecture\n\nThe architectural con...


In [114]:
test_df.to_csv("synthethic_data.csv")

In [115]:
!pwd

/content


# Rouge Score

Ragas Documentation Rouge Score: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/traditional/?h=rouge#rouge-score

In [89]:
!pip install --quiet rouge-score

  Preparing metadata (setup.py) ... done


In [106]:
# Modern Version 6/29/2026
from ragas.metrics.collections import RougeScore

# Create metric (no LLM/embeddings needed)
scorer = RougeScore(rouge_type="rougeL", mode="fmeasure")

# Evaluate
result = await scorer.ascore(
    reference="The Eiffel Tower is located in Paris.",
    response="The Eiffel Tower is located in India."
)
print(f"ROUGE Score: {result.value}")

ROUGE Score: 0.8571428571428571


In [105]:
# Simple sample of RougeScore
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import RougeScore

sample = SingleTurnSample(
    response = "The Eiffel Tower in located in India.",
    reference = "The Eiffel Tower in located in Paris."
)

scorer = RougeScore()
score = await scorer.single_turn_ascore(sample)
print(score)

0.8571428571428571


/tmp/ipykernel_3077/2714973045.py:3: DeprecationWarning: Importing RougeScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import RougeScore
  from ragas.metrics import RougeScore


In [107]:
# Import classes and libraries
from ragas.metrics.collections import RougeScore
import numpy as np

# Create metric (no LLM/embeddings needed)
scorer = RougeScore(rouge_type="rougeL", mode="fmeasure")

In [112]:
rouge_score_list = []
for row in test_df.to_dict(orient = "records"):
  rouge_score = await scorer.ascore(
      reference = row["reference"],
      response = row["answer_gen"]
  )
  rouge_score_list.append(rouge_score.value)

In [117]:
print(f"First 5 rouge scores: {rouge_score_list[0:5]}")
print(f"The mean rouge score is: {np.mean(rouge_score_list)}")

First 5 rouge scores: [0.3054755043227666, 0.33962264150943394, 0.14437869822485208, 0.06844919786096255, 0.1785714285714286]
The mean rouge score is: 0.16135228533845392


In [119]:
import ragas
print(ragas.__version__)

0.4.3


# LLM Based Evaluation

In [130]:
# Define the evaluation llm and embedding model
from openai import AsyncOpenAI, OpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory

# Connect to OpenAI
client = AsyncOpenAI(api_key = openai_api)
#client = OpenAI(api_key = openai_api)

eval_llm = llm_factory(
    model = "gpt-4o-mini",
    client = client
)

print(f"[INFO] Our embedding model is: {EMBEDDING_MODEL}")
eval_embeddings = embedding_factory(
    "openai",
    model = EMBEDDING_MODEL,
    client = client
)

[INFO] Our embedding model is: text-embedding-3-large


## Simple Criteria Scoring
Simple Criteria Scoring RAGAS: https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/general_purpose/#simple-criteria-scoring

In [131]:
from ragas.metrics import DiscreteMetric

In [133]:
# Define the simple scorer
simple_scorer = DiscreteMetric(
    name="simple scorer",
    allowed_values=list(range(0, 5)),  # 0 to 5
    prompt="""Score 0 to 5 by similarity between reference and generated answer

    The following are the keys:
    user_input : {user_input}
    reference : {reference}
    generated answer : {response}

    Respond with only the number (0-5).
    """,
)


In [134]:
# Get the simple score for every row
simple_score_list = []
for row in test_df.to_dict(orient = "records"):
  simple_score = await simple_scorer.ascore(
      user_input = row["user_input"],
      reference = row["reference"],
      response = row["answer_gen"],
      llm = eval_llm
  )
  simple_score_list.append(simple_score.value)

In [135]:
# Print the outputs
print(simple_score_list)
print(f"The mean simple score is: {np.mean(simple_score_list)}")

[4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 4, 4, 4, 3, 4, 4, 4, 4, 4]
The mean simple score is: 3.875


## Rubrics Based Scoring

Link: https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/general_purpose/#rubrics-based-scoring

In [136]:
rubrics = {
    "score1_description": "The response is entirely incorrect and fails to address any aspect of the reference.",
    "score2_description": "The response contains partial accuracy but includes major errors or significant omissions that affect its relevance to the reference.",
    "score3_description": "The response is mostly accurate but lacks clarity, thoroughness, or minor details needed to fully address the reference.",
    "score4_description": "The response is accurate and clear, with only minor omissions or slight inaccuracies in addressing the reference.",
    "score5_description": "The response is completely accurate, clear, and thoroughly addresses the reference without any errors or omissions.",
}

In [142]:
from ragas.metrics.collections import RubricsScoreWithReference

In [146]:
# Define the rubric Scorer
rubrics_scorer = RubricsScoreWithReference(
    rubrics=rubrics,
    prompt="""Using the rubrics passed, score the reference and generated answer

    The following are the keys:
    user_input : {user_input}
    reference : {reference}
    generated answer : {response}
    """,
    llm=eval_llm,
)


In [147]:
# Compute the rubrics score for each row
rubrics_score_list = []
for row in tqdm(test_df.to_dict(orient = "records"), desc = "Computing Rubrics Score...", total = len(test_df.to_dict(orient = "records"))):
  rubrics_score = await rubrics_scorer.ascore(
      user_input = row["user_input"],
      reference = row["reference"],
      response = row["answer_gen"]
  )
  rubrics_score_list.append(rubrics_score.value)

Computing Rubrics Score...: 100%|██████████| 32/32 [01:10<00:00,  2.19s/it]


In [148]:
# Print the rubrics score using the score
print(rubrics_score_list)
print(f"The mean rubrics score is: {np.mean(rubrics_score_list)}")

[4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 5.0, 5.0, 4.0, 5.0, 4.0, 4.0, 4.0, 4.0, 4.0, 5.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 5.0, 4.0, 4.0, 4.0, 4.0]
The mean rubrics score is: 4.15625


# RAG-Specific Metrics

## Factual Correctness

In [149]:
from ragas.metrics.collections import FactualCorrectness

In [171]:
# Decfine the Factual Correcteness scorer
# Connect to OpenAI
client = AsyncOpenAI(api_key = openai_api)
#client = OpenAI(api_key = openai_api)

eval_llm = llm_factory(
    model = "gpt-4o-mini",
    client = client
)

# eval_llm = llm_factory(
#     model = "gpt-5-mini",
#     client = client,
#     max_tokens = 2048
# )

factual_scorer = FactualCorrectness(
    llm=eval_llm,
    atomicity="low",
    coverage="low")

In [166]:
factual_scores_list = []
for row in tqdm(test_df.to_dict(orient = "records"), desc = "Computing Factual Score...", total = len(test_df.to_dict(orient = "records"))):
  factual_score = await factual_scorer.ascore(
      reference = row["reference"],
      response = row["answer_gen"]
  )
  factual_scores_list.append(factual_score.value)

Computing Factual Score...:   6%|▋         | 2/32 [02:00<30:06, 60.22s/it]


IncompleteOutputException: The output is incomplete due to a max_tokens length limit.

In [ ]:
# Print the first 5 and mean
print(factual_scores_list[0:5])
print(f"The mean factual score is: {np.mean(factual_scores_list)}")

## Semantic Similarity

In [167]:
from ragas.metrics.collections import SemanticSimilarity

In [168]:
# Get the semantic scorer
semantic_scorer = SemanticSimilarity(
    embeddings = eval_embeddings)

In [169]:
semantic_similarity_score = []
for row in tqdm(test_df.to_dict(orient = "records"), desc = "Computing Semantic Similarity Score...", total = len(test_df.to_dict(orient = "records"))):
  semantic_score = await semantic_scorer.ascore(
      reference = row["reference"],
      response = row["answer_gen"]
  )
  semantic_similarity_score.append(semantic_score.value)


Computing Semantic Similarity Score...: 100%|██████████| 32/32 [01:33<00:00,  2.92s/it]


In [170]:
# Print the first 5 and mean
print(semantic_similarity_score[0:5])
print(f"The mean semantic similarity score is: {np.mean(semantic_similarity_score)}")

[0.8228008285737077, 0.8390561230990111, 0.8145773360878539, 0.7506930152047856, 0.729742946449343]
The mean semantic similarity score is: 0.7955267543073357


## Context Precision

In [180]:
from ragas.metrics.collections import ContextPrecision

In [181]:
client = AsyncOpenAI(api_key = openai_api)
#client = OpenAI(api_key = openai_api)

eval_llm = llm_factory(
    model = "gpt-4o-mini",
    client = client
)

In [182]:
context_precision_scorer = ContextPrecision(llm = eval_llm)

In [183]:
# Create a context precision list
context_precision_scores = []
for row in tqdm(test_df.to_dict(orient = "records"), desc = "Computing Context Precision Score...", total = len(test_df.to_dict(orient = "records"))):
  context_precision_score = await context_precision_scorer.ascore(
      user_input = row["user_input"],
      reference = row["reference"],
      retrieved_contexts=row["context_gen"])
  context_precision_scores.append(context_precision_score.value)

Computing Context Precision Score...: 100%|██████████| 32/32 [11:36<00:00, 21.77s/it]


In [ ]:
# Print the first 5 and mean
print(context_precision_scores[0:5])
print(f"The mean context precision score is: {np.mean(context_precision_scores)}")

## Context Recall

In [172]:
from ragas.metrics.collections import ContextRecall

In [173]:
client = AsyncOpenAI(api_key = openai_api)
#client = OpenAI(api_key = openai_api)

eval_llm = llm_factory(
    model = "gpt-4o-mini",
    client = client
)

In [174]:
# Setup context recall scorer
context_recall_scorer = ContextRecall(llm = eval_llm)

In [177]:
# Create a context recall list
context_recall_scores = []
for row in tqdm(test_df.to_dict(orient = "records"), desc = "Computing Context Recall Score...", total = len(test_df.to_dict(orient = "records"))):
  context_recall_score = await context_recall_scorer.ascore(
      user_input = row["user_input"],
      reference = row["reference"],
      retrieved_contexts=row["context_gen"]
  )
  context_recall_scores.append(context_recall_score.value)

Computing Context Recall Score...: 100%|██████████| 32/32 [03:01<00:00,  5.66s/it]


In [179]:
# Print the first 5 resutls and mean
print(context_recall_scores[0:5])
print(f"The mean context recall score is: {np.mean(context_recall_scores)}")

[1.0, 1.0, 1.0, 1.0, 1.0]
The mean context recall score is: 0.9692708333333333


## Response Relevancy

In [184]:
from ragas.metrics.collections import AnswerRelevancy

In [185]:
# Define the response relevancy scorer
client = AsyncOpenAI(api_key = openai_api)
#client = OpenAI(api_key = openai_api)

eval_llm = llm_factory(
    model = "gpt-4o-mini",
    client = client)

eval_embeddings = embedding_factory(
    "openai",
    model = EMBEDDING_MODEL,
    client = client
)

response_relevancy_scorer = AnswerRelevancy(
    llm = eval_llm,
    embeddings = eval_embeddings)

In [188]:
# Compute the response relevancies scores
response_relevancy_scores = []
for row in tqdm(test_df.to_dict(orient = "records"), desc = "Computing Response Relevancy Score...", total = len(test_df.to_dict(orient = "records"))):
  response_relevancy_score = await response_relevancy_scorer.ascore(
      user_input= row["user_input"],
      response = row["answer_gen"]
  )
  response_relevancy_scores.append(response_relevancy_score.value)

Computing Response Relevancy Score...: 100%|██████████| 32/32 [03:01<00:00,  5.66s/it]
